# Formula E — Camera mapping & coverage check

Purpose: lock down which mosaic **group** and **panel** each telemetry incident really maps to, *before* we build the telemetry-triggered video **verifier**.

**Why this matters:** the GPS→camera projection currently disagrees with some of our turn *labels* (e.g. Günther is labeled `T1-2` but his GPS projects near **T13 / Cam19 / Group 5**). This notebook shows the actual mosaic frame at each incident's time so we can confirm, by eye, which camera truly sees each stopped car — and whether the 2×2 panels are sharp enough, or we need the full-res single-camera fallback.

Run top-to-bottom. Needs this project's ADC (default in Colab Enterprise) with **read on the mosaics bucket**.

## 1 · Config

In [ ]:
PROJECT_ID = ""   # e.g. "qwiklabs-gcp-04-..."; blank = notebook default project

# Where the staged 2x2 group mosaics live (each is <group_id>.mp4, full race @ 1 FPS):
MOSAICS_BASE = "gs://class-demo/formula-e/r10/mosaics"
#   ... or your project copy:  f"gs://{PROJECT_ID}-fe-mosaics/mosaics"

WORK = "/content/fe_check"
import os; os.makedirs(WORK, exist_ok=True)
print("mosaics base:", MOSAICS_BASE)

## 2 · Setup

In [ ]:
import subprocess, sys, json, glob
try:
    from google.cloud import storage
except Exception:
    subprocess.run([sys.executable,"-m","pip","install","-q","google-cloud-storage"], check=True)
    from google.cloud import storage
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
print("ffmpeg:", subprocess.run(["ffmpeg","-version"],capture_output=True,text=True).stdout.splitlines()[0])

## 3 · Geometry, groups & the incidents to check
(bounds + cameras pulled straight from `track_geometry.json`)

In [ ]:
BOUNDS = {"lat_min": 52.477768, "lat_max": 52.482359, "lng_min": 13.388477, "lng_max": 13.397453, "pad": 30, "w": 1000, "h": 620}
CAMERAS = [{"id": "Cam01", "turn": "FL", "x": 747.2, "y": 123.0}, {"id": "Cam02", "turn": "FL/T1", "x": 617.4, "y": 149.7}, {"id": "Cam03", "turn": "T1", "x": 444.2, "y": 192.7}, {"id": "Cam04", "turn": "T2", "x": 309.0, "y": 300.9}, {"id": "Cam05", "turn": "T2/T3", "x": 220.4, "y": 400.4}, {"id": "Cam06", "turn": "T3", "x": 145.2, "y": 402.6}, {"id": "Cam07", "turn": "T4", "x": 200.1, "y": 319.7}, {"id": "Cam08", "turn": "T4/T5", "x": 317.4, "y": 233.4}, {"id": "Cam09", "turn": "T5/T6", "x": 412.4, "y": 182.9}, {"id": "Cam10", "turn": "T6", "x": 370.8, "y": 156.4}, {"id": "Cam11", "turn": "T7", "x": 234.9, "y": 231.4}, {"id": "Cam12", "turn": "T7/T8", "x": 77.6, "y": 408.4}, {"id": "Cam13", "turn": "T9A", "x": 50.8, "y": 562.3}, {"id": "Cam14", "turn": "T9B", "x": 160.6, "y": 581.5}, {"id": "Cam15", "turn": "T10", "x": 257.3, "y": 469.7}, {"id": "Cam16", "turn": "T11", "x": 487.6, "y": 312.7}, {"id": "Cam17", "turn": "T11/T12", "x": 832.3, "y": 152.6}, {"id": "Cam18", "turn": "T12", "x": 941.4, "y": 73.6}, {"id": "Cam19", "turn": "T13", "x": 878.9, "y": 36.6}, {"id": "Cam20", "turn": "T14", "x": 750.2, "y": 47.9}, {"id": "Cam21", "turn": "PIT ENTRY", "x": 756.6, "y": 106.0}, {"id": "Cam22", "turn": "PIT EXIT", "x": 769.2, "y": 157.0}, {"id": "Cam23", "turn": "PIT A", "x": 791.2, "y": 157.0}, {"id": "Cam24", "turn": "PIT B", "x": 813.2, "y": 157.0}]
GROUPS = {"grp_01_cam01_cam02_cam03_cam04": ["Cam01", "Cam02", "Cam03", "Cam04"], "grp_02_cam05_cam06_cam07_cam08": ["Cam05", "Cam06", "Cam07", "Cam08"], "grp_03_cam09_cam10_cam11_cam12": ["Cam09", "Cam10", "Cam11", "Cam12"], "grp_04_cam13_cam14_cam15_cam16": ["Cam13", "Cam14", "Cam15", "Cam16"], "grp_05_cam17_cam18_cam19_cam20": ["Cam17", "Cam18", "Cam19", "Cam20"], "grp_06_cam21_cam22_cam23_cam24": ["Cam21", "Cam22", "Cam23", "Cam24"]}
INCIDENTS = [{"label": "G\u00fcnther #7 (labeled T1-2)", "t": 693, "lat": 52.482365, "lng": 13.396783}, {"label": "Nato #17 (labeled T15)", "t": 1692, "lat": 52.481997, "lng": 13.395208}, {"label": "Fenestraz #23 (labeled T15)", "t": 1692, "lat": 52.482001, "lng": 13.395187}, {"label": "Mortara #48", "t": 1780, "lat": 52.479635, "lng": 13.391701}]
CAM_BY_ID = {c['id']: c for c in CAMERAS}
print(len(CAMERAS), 'cameras,', len(GROUPS), 'groups,', len(INCIDENTS), 'incidents')

## 4 · GPS → camera / group / panel  (the hypothesis)

In [ ]:
def gpsXY(lat, lng):
    B = BOUNDS
    x = B['pad'] + (lng-B['lng_min'])/(B['lng_max']-B['lng_min'])*(B['w']-2*B['pad'])
    y = B['pad'] + (B['lat_max']-lat)/(B['lat_max']-B['lat_min'])*(B['h']-2*B['pad'])
    return x, y

def group_of(cam_id):
    for gid, cams in GROUPS.items():
        if cam_id in cams:
            return gid, cams.index(cam_id)
    return None, None

def nearest_cams(lat, lng, k=3):
    x, y = gpsXY(lat, lng)
    return sorted(CAMERAS, key=lambda c:(c['x']-x)**2+(c['y']-y)**2)[:k]

print(f"{'incident':30}{'proj(x,y)':13}{'nearest cam':16}{'group':34}{'panel'}")
for i in INCIDENTS:
    x, y = gpsXY(i['lat'], i['lng'])
    c = nearest_cams(i['lat'], i['lng'])[0]
    gid, panel = group_of(c['id'])
    print(f"{i['label']:30}{f'({x:.0f},{y:.0f})':13}{c['id']+' '+c['turn']:16}{gid:34}{panel}")

## 5 · Map view — cameras (in track order) + incident projections
If an incident star sits far from its label's turn, the label (or the camera coords) is off.

In [ ]:
fig, ax = plt.subplots(figsize=(12,7))
xs = [c['x'] for c in CAMERAS]; ys = [c['y'] for c in CAMERAS]
ax.plot(xs+xs[:1], ys+ys[:1], '-', color='#ccc', lw=1, zorder=1)
for c in CAMERAS:
    ax.scatter(c['x'], c['y'], s=18, color='#37475a', zorder=2)
    ax.annotate(f"{c['id']} {c['turn']}", (c['x'], c['y']), fontsize=6, ha='center', va='bottom')
for i in INCIDENTS:
    x, y = gpsXY(i['lat'], i['lng'])
    ax.scatter(x, y, s=120, color='#ff4d4d', marker='*', zorder=3)
    ax.annotate(i['label'], (x, y), fontsize=8, color='#b00', ha='left', va='top')
ax.set_ylim(ax.get_ylim()[::-1]); ax.set_aspect('equal')
ax.set_title('Cameras (connected in track order) + incident GPS projections'); plt.show()

## 6 · Mosaic fetch + single-frame extract
(mosaic is 1 FPS, so race-second N == frame N)

In [ ]:
_gcs = storage.Client(project=PROJECT_ID or None)

def fetch_mosaic(group_id):
    local = os.path.join(WORK, f'{group_id}.mp4')
    if not os.path.exists(local):
        gs = f'{MOSAICS_BASE}/{group_id}.mp4'
        bkt = gs.split('/')[2]; key = '/'.join(gs.split('/')[3:])
        print('downloading', gs)
        _gcs.bucket(bkt).blob(key).download_to_filename(local)
    return local

def frame_at(group_id, sec):
    mp4 = fetch_mosaic(group_id)
    out = os.path.join(WORK, f'{group_id}_{sec}.jpg')
    subprocess.run(['ffmpeg','-y','-loglevel','error','-ss',str(sec),'-i',mp4,
                    '-frames:v','1','-q:v','2',out], check=True)
    return Image.open(out)

## 7 · Show a group frame with the panels labelled (and one boxed)

In [ ]:
def show_group(group_id, sec, highlight=None, ax=None):
    img = frame_at(group_id, sec); W, H = img.size
    own = ax is None
    if own: fig, ax = plt.subplots(figsize=(7,5))
    ax.imshow(img)
    cams = GROUPS[group_id]
    quads = [(0,0),(W/2,0),(0,H/2),(W/2,H/2)]   # TL, TR, BL, BR == panels 0..3
    for idx,(qx,qy) in enumerate(quads):
        ax.add_patch(patches.Rectangle((qx,qy), W/2, H/2, fill=False, edgecolor='#888', lw=0.5))
        ax.annotate(cams[idx], (qx+6, qy+16), color='yellow', fontsize=9, weight='bold')
    if highlight is not None:
        qx, qy = quads[highlight]
        ax.add_patch(patches.Rectangle((qx,qy), W/2, H/2, fill=False, edgecolor='#ff3', lw=3))
    ax.set_title(f'{group_id}\n t={sec}s   frame {W}x{H}, panel {W//2}x{H//2}', fontsize=9)
    ax.axis('off')
    if own: plt.show()

## 8 · Validate each incident
For each incident: the GPS-nearest group at the stop time and +10s, with the picked panel boxed. **Is the stopped car actually in the boxed panel?**

In [ ]:
for i in INCIDENTS:
    c = nearest_cams(i['lat'], i['lng'])[0]; gid, panel = group_of(c['id'])
    print(f"\n### {i['label']}  ->  nearest {c['id']} ({c['turn']}), {gid}, panel {panel}")
    fig, axs = plt.subplots(1, 2, figsize=(15,5))
    show_group(gid, i['t'],    highlight=panel, ax=axs[0])
    show_group(gid, i['t']+10, highlight=panel, ax=axs[1])
    plt.show()

## 9 · Manual compare — test a label against the GPS pick
Günther is *labeled* T1-2 (Group 1) but *projects* to Group 5. Look at both at t=693 and see which one actually has a stopped car.

In [ ]:
show_group('grp_01_cam01_cam02_cam03_cam04', 693)   # the label's group (T1-2)
show_group('grp_05_cam17_cam18_cam19_cam20', 693)   # the GPS pick (T13/T14)

## 10 · Temporal / dead-panel coverage
Samples each group every 60s and flags panels that are ~black (a camera with no footage there). Downloads all six mosaics (~350 MB).

In [ ]:
def panel_coverage(group_id, step=60):
    mp4 = fetch_mosaic(group_id)
    d = os.path.join(WORK, f'{group_id}_scan'); os.makedirs(d, exist_ok=True)
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',mp4,'-vf',f'fps=1/{step}',
                    os.path.join(d,'f%04d.jpg')], check=True)
    files = sorted(glob.glob(os.path.join(d,'f*.jpg')))
    means = [[],[],[],[]]
    for f in files:
        im = np.asarray(Image.open(f).convert('L'), dtype=float); H, W = im.shape
        q = [im[:H//2,:W//2], im[:H//2,W//2:], im[H//2:,:W//2], im[H//2:,W//2:]]
        for p in range(4): means[p].append(q[p].mean())
    return files, means

for gid in GROUPS:
    files, means = panel_coverage(gid)
    cams = GROUPS[gid]
    dead = [cams[p] for p in range(4) if np.mean(means[p]) < 12]
    print(f"{gid}: {len(files)} samples over the race   dead/near-black panels: {dead or 'none'}")

## 11 · Resolution readout

In [ ]:
img = frame_at(next(iter(GROUPS)), 693); W, H = img.size
print(f'mosaic frame {W}x{H}  ->  each camera panel is {W//2}x{H//2}')
print('If a distant car is unreadable at that panel size, that argues for the full-res single-camera fallback.')

## 12 · The real test — let **Gemini** watch a slice

Instead of eyeballing frames, prototype the actual verifier: slice a ~45s window at the incident time and ask Gemini to CONFIRM or DENY the stopped car. This is the exact mechanism we'll ship, and it doubles as the mapping check — if Gemini finds the car in the group we mapped, the mapping is right; if it says NOT CONFIRMED there but confirms in another group, we relabel.

A window (not a single frame) also lets Gemini judge **stopped vs frozen feed**: a genuinely stopped car has other cars moving past it; a dead camera is frozen everywhere.

In [ ]:
# Gemini client (Vertex + ADC, global endpoint — no API key), matching the app.
try:
    from google import genai
    from google.genai import types
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','google-genai'], check=True)
    from google import genai
    from google.genai import types
import google.auth
_proj = PROJECT_ID or google.auth.default()[1]
GEM = genai.Client(vertexai=True, project=_proj, location='global')
GEM_MODEL = 'gemini-3.5-flash'
print('Gemini ready via Vertex project', _proj, 'model', GEM_MODEL)

In [ ]:
# Slice a window of frames from a group's mosaic; optionally crop to one panel.
def window_frames(group_id, t, window=45, step=3, panel=None):
    secs = list(range(t, t + window + 1, step))
    files = []
    for s in secs:
        img = frame_at(group_id, s)
        if panel is not None:
            W, H = img.size
            qx, qy = [(0,0),(W/2,0),(0,H/2),(W/2,H/2)][panel]
            img = img.crop((qx, qy, qx + W/2, qy + H/2))
        fp = os.path.join(WORK, f"{group_id}_{s}_{'p'+str(panel) if panel is not None else 'full'}.jpg")
        img.save(fp); files.append(fp)
    return files

In [ ]:
import re
POS = ['TL','TR','BL','BR']

def verify(incident, group_id=None, crop=False, window=45, step=3):
    """Prototype verifier: send a window to Gemini, print CONFIRM/DENY.
    group_id=None -> use the GPS-mapped group. crop=True -> single mapped panel."""
    c = nearest_cams(incident['lat'], incident['lng'])[0]
    gid_auto, panel = group_of(c['id'])
    gid = group_id or gid_auto
    cams = GROUPS[gid]
    use_panel = panel if (crop and group_id is None) else None
    files = window_frames(gid, incident['t'], window, step, panel=use_panel)
    m = re.search(r'#(\d+)', incident['label']); car = m.group(1) if m else '?'
    turn = c['turn']
    schema = ('Respond JSON: {"confirmed": bool, "panel": "TL|TR|BL|BR|none", '
              '"feed_live": bool, "what_you_see": str, "confidence": number}')
    if use_panel is not None:
        where = f'These {len(files)} frames (1 every {step}s over ~{window}s) are from ONE CCTV camera {c["id"]} ({turn}).'
        schema = 'Respond JSON: {"confirmed": bool, "feed_live": bool, "what_you_see": str, "confidence": number}'
    else:
        where = (f'These {len(files)} frames (1 every {step}s over ~{window}s) are a 2x2 CCTV mosaic: '
                 f'TL={cams[0]}, TR={cams[1]}, BL={cams[2]}, BR={cams[3]}. '
                 f'The stop is expected in the {POS[panel]} panel ({cams[panel]}), but check all four.')
    prompt = (
        'You are a race-control video verifier. Telemetry flagged a possible incident; confirm or deny it from CCTV.\n'
        f'Telemetry report: car #{car} appears STOPPED on track near {turn}, at race time ~{incident["t"]}s.\n'
        f'{where}\n'
        'Judge: (1) is a stopped/stranded car or on-track hazard clearly visible, consistent with the report? '
        '(2) are OTHER cars moving across the frames (feed is live) or is everything frozen (a dead camera feed)? '
        'If you cannot clearly see a stopped car, answer confirmed=false.\n' + schema)
    parts = [types.Part.from_bytes(data=open(f,'rb').read(), mime_type='image/jpeg') for f in files]
    parts.append(types.Part(text=prompt))
    resp = GEM.models.generate_content(
        model=GEM_MODEL, contents=[types.Content(role='user', parts=parts)],
        config=types.GenerateContentConfig(temperature=0.2, response_mime_type='application/json'))
    tag = ('crop:'+c['id']) if use_panel is not None else 'mosaic'
    print(f"=== {incident['label']}  |  group={gid}  |  {tag}  |  {len(files)} frames ===")
    print(resp.text, '\n')
    return resp.text

### 12a · Verify each incident on its GPS-mapped group (mosaic window)

In [ ]:
for i in INCIDENTS:
    verify(i)

### 12b · Günther A/B — label group vs GPS group
Günther is *labeled* T1-2 (Group 1) but *projects* to Group 5. Ask Gemini both — whichever it confirms is the truth.

In [ ]:
g = INCIDENTS[0]
verify(g, group_id='grp_01_cam01_cam02_cam03_cam04')   # the label's group
verify(g, group_id='grp_05_cam17_cam18_cam19_cam20')   # the GPS pick

### 12c · Mosaic vs single-camera crop
Does cropping to the one mapped panel help or hurt? (full 2x2 keeps neighbour context; crop is a cleaner but lower-res single view.)

In [ ]:
verify(INCIDENTS[0])              # full 2x2 mosaic
verify(INCIDENTS[0], crop=True)   # single mapped panel only

### Read-out
- If Gemini **confirms** on the GPS group and **denies** on the label group, the labels were wrong and our earlier 'corroboration' was hallucinated — we relabel each incident to its GPS group.
- If it denies everywhere, the mosaic panels are too low-res / the car is off-camera → full-res single-camera fallback.
- `feed_live=false` on a confirm means we caught a frozen/dead panel — note it for that camera.